# DATA ANALYTICS ML CHALLENGE / 2026-2
## Laboratorio 1. Modelos supervisados y métricas (Semana 12)

Profesor: Elias Buitrago Bolivar  
Universidad Central  
Caso: Brazilian E-Commerce Public Dataset by Olist

Objetivo: construir un modelo base para anticipar entregas tardías a partir de datos de compras, productos, clientes, vendedores y pagos.

Las instrucciones de GitHub y entrega se encuentran en el documento técnico. Las secciones de carga y preparación se suministran resueltas; complete las cuatro actividades de modelado y evaluación.

## 1. Carga de datos

Una observación del modelo corresponde a un pedido, aunque tenga varios productos y pagos. Se integrarán siete tablas del conjunto público de Olist.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

In [2]:
ORIGEN = ('https://raw.githubusercontent.com/spdrio/'
          'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
          'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')

pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')

print('Pedidos:', pedidos.shape)
print('Items:', items.shape)
print('Productos:', productos.shape)
print('Clientes:', clientes.shape)
print('Vendedores:', vendedores.shape)
print('Pagos:', pagos.shape)

Pedidos: (99441, 8)
Items: (112650, 7)
Productos: (32951, 9)
Clientes: (99441, 5)
Vendedores: (3095, 4)
Pagos: (103886, 5)


## 2. Preparación de variables

Un pedido puede contener varios ítems y pagos. Primero se resumen estas tablas por `order_id`, para evitar contar un mismo pedido varias veces. Las variables de destino se construyen después y no se incluyen entre los predictores.

In [3]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [4]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

Pedidos integrados: 98666


In [5]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

C:\Users\cifue\AppData\Local\Temp\ipykernel_12596\3678014271.py:16: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  data['dia_semana_compra'] = compra.dt.dayofweek


Pedidos disponibles: 96470
Entregas tardías: 6.77 %


### Variables del modelo

Se utilizarán datos que razonablemente están disponibles al cierre de la compra: plazo prometido, calendario, contenido del pedido, dimensiones de los productos, ubicación y medio de pago. Las fechas reales de envío y entrega, y las reseñas posteriores, quedan excluidas de las variables predictoras.

La lista de variables y sus tablas de origen se encuentra en el diccionario técnico anexo.

In [6]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

Variables predictoras: 28
Numéricas: 24 | Categóricas: 4


### Muestra individual

Cada estudiante utilizará 35.000 pedidos, seleccionados a partir del ID numérico entregado por el profesor (por ejemplo, ID 01). El mismo ID genera siempre la misma muestra y las mismas particiones.

Se utilizarán 24.500 pedidos para entrenar, 5.250 para validar y 5.250 para la prueba final. Durante las semanas 12 a 15 no se utilizarán los datos de prueba para elegir modelos.

En la siguiente celda cambie `ID_ESTUDIANTE` por el número asignado por el profesor. Por ejemplo, el ID 01 se escribe como `1`.


In [7]:
# Escriba el ID que le asignó el profesor (01 se escribe 1).
ID_ESTUDIANTE = 26
assert isinstance(ID_ESTUDIANTE, int) and 1 <= ID_ESTUDIANTE <= 30, (
    'El ID debe ser un número entero entre 1 y 30.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)

Estudiante ID: 26 | Semilla: 1026
Muestra: 35000 | Train: 24500 | Valid: 5250 | Test: 5250
Huella de muestra: 78a4ad412176


## 3. Modelo de referencia y clasificación

Las variables numéricas se imputan y escalan; las categóricas se transforman con *One-Hot Encoding*. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, dentro del `Pipeline`.

In [8]:
preprocesamiento = ColumnTransformer([
    ('numericas', make_pipeline(SimpleImputer(strategy='median'),
                               StandardScaler()), columnas_numericas),
    ('categoricas', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')),
     columnas_categoricas)
])

referencia = DummyClassifier(strategy='most_frequent')
referencia.fit(X_train, y_train)
pred_ref = referencia.predict(X_valid)

print('Accuracy referencia:', round(accuracy_score(y_valid, pred_ref), 3))
print('F1 referencia:', round(f1_score(y_valid, pred_ref, zero_division=0), 3))

Accuracy referencia: 0.932
F1 referencia: 0.0


### Actividad 1. Entrenamiento

Complete el entrenamiento de la regresión logística usando `preprocesamiento`. Pruebe `class_weight=None` o `'balanced'`, y explique su elección en la sección de interpretación. Conserve las variables `modelo` y `pred`.

En el estimador utilice `class_weight=peso_clases` y `max_iter=1000` como límite de iteraciones. Si aparece una advertencia de convergencia, revise el entrenamiento.


In [9]:
# Actividad 1. Entrenamiento de la regresión logística.
peso_clases = 'balanced'  # Opción: None

modelo = make_pipeline(
    preprocesamiento,
    LogisticRegression(class_weight=peso_clases, max_iter=1000, random_state=semilla)
)
modelo.fit(X_train, y_train)
pred = modelo.predict(X_valid)
print('Predicciones de validación:', len(pred), '| tardías predichas:', int(pred.sum()))

Predicciones de validación: 5250 | tardías predichas: 1883


## 4. Evaluación

### Actividad 2. Métricas

Calcule Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (AP) en validación. Para las métricas AUC use las probabilidades que estima el modelo.

In [10]:
# Actividad 2. Métricas en validación.
proba = modelo.predict_proba(X_valid)[:, 1]
medidas = {
    'accuracy': accuracy_score(y_valid, pred),
    'precision': precision_score(y_valid, pred, zero_division=0),
    'recall': recall_score(y_valid, pred),
    'f1': f1_score(y_valid, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, proba),
    'ap': average_precision_score(y_valid, proba),
}
print(pd.Series(medidas).round(3).to_string())

# Comparación con el modelo de referencia (siempre "a tiempo").
print('\nReferencia -> Accuracy:', round(accuracy_score(y_valid, pred_ref), 3),
      '| Recall:', round(recall_score(y_valid, pred_ref, zero_division=0), 3),
      '| F1:', round(f1_score(y_valid, pred_ref, zero_division=0), 3))

cm = confusion_matrix(y_valid, pred)
print('\nMatriz de confusión (filas = real, columnas = predicho)')
print(pd.DataFrame(cm, index=['Real a tiempo', 'Real tardío'],
                   columns=['Pred a tiempo', 'Pred tardío']))

accuracy     0.661
precision    0.122
recall       0.645
f1           0.205
roc_auc      0.705
ap           0.145

Referencia -> Accuracy: 0.932 | Recall: 0.0 | F1: 0.0

Matriz de confusión (filas = real, columnas = predicho)
               Pred a tiempo  Pred tardío
Real a tiempo           3241         1654
Real tardío              126          229


### Actividad 3. Errores por segmento

Compare la tasa de error por mes de compra y estado del cliente. Muestre tanto el porcentaje de error como la cantidad de pedidos de cada segmento.

In [11]:
assert pred is not None, 'Complete primero la Actividad 1 (variable pred).'
analisis = valid.copy()
analisis['pred'] = pred
analisis['error'] = (analisis['is_late'] != analisis['pred']).astype(int)

# Actividad 3. Error por mes de compra y por estado del cliente.
def tabla_error(df, columna):
    t = df.groupby(columna).agg(pedidos=('error', 'size'),
                                errores=('error', 'sum'),
                                tasa_error_pct=('error', lambda s: 100 * s.mean()),
                                tardios_reales=('is_late', 'sum'))
    t['tasa_error_pct'] = t['tasa_error_pct'].round(1)
    return t

error_mes = tabla_error(analisis, 'mes_compra')
error_estado = tabla_error(analisis, 'customer_state').sort_values('tasa_error_pct', ascending=False)

print('Error por mes de compra')
print(error_mes.to_string())
print('\nError por estado del cliente (estados con al menos 100 pedidos)')
print(error_estado[error_estado['pedidos'] >= 100].to_string())

Error por mes de compra
            pedidos  errores  tasa_error_pct  tardios_reales
mes_compra                                                  
1               422      139            32.9              24
2               454      170            37.4              55
3               523      216            41.3              83
4               499      144            28.9              36
5               565      222            39.3              29
6               495      110            22.2              11
7               561      222            39.6              18
8               544      259            47.6              17
9               203       73            36.0               6
10              256       63            24.6               7
11              416      124            29.8              53
12              312       38            12.2              16

Error por estado del cliente (estados con al menos 100 pedidos)
                pedidos  errores  tasa_error_pct  tardios

### Actividad 4. Costo empresarial

Para el ejercicio, suponga un costo de R$ 8 por falso positivo y R$ 35 por falso negativo. Calcule el costo total de los errores en validación.

In [12]:
# Actividad 4. Costo empresarial: R$ 8 por falso positivo y R$ 35 por falso negativo.
tn, fp, fn, tp = confusion_matrix(y_valid, pred).ravel()
costo_fp_fn = 8 * fp + 35 * fn
print(f'Falsos positivos: {fp} | Falsos negativos: {fn}')
print(f'Costo total de errores (validación): R$ {costo_fp_fn:,.0f}')

# Comparación: costo del modelo de referencia (nunca avisa de un retraso).
tn_r, fp_r, fn_r, tp_r = confusion_matrix(y_valid, pred_ref).ravel()
print(f'Costo del modelo de referencia: R$ {8 * fp_r + 35 * fn_r:,.0f}')

Falsos positivos: 1654 | Falsos negativos: 126
Costo total de errores (validación): R$ 17,642
Costo del modelo de referencia: R$ 12,425


In [13]:
# Prueba de umbrales en validación (modelo con class_weight='balanced')
resultados = []
for umbral in np.arange(0.05, 0.96, 0.01):
    p = (proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, p).ravel()
    resultados.append((round(umbral, 2), fp, fn, 8 * fp + 35 * fn))
tabla = pd.DataFrame(resultados, columns=['umbral', 'FP', 'FN', 'costo'])
print(tabla[tabla['umbral'].isin([0.19, 0.30, 0.50, 0.60, 0.70])].to_string(index=False))
print('\nUmbral con menor costo:')
print(tabla.loc[tabla['costo'].idxmin()].to_string())

 umbral   FP  FN  costo
   0.19 4388   5  35279
   0.30 3637  23  29901
   0.50 1654 126  17642
   0.60  923 195  14209
   0.70  398 263  12389

Umbral con menor costo:
umbral        0.76
FP          209.00
FN          302.00
costo     12242.00


In [14]:
# Evidencia para la pregunta 1: modelo sin balancear (class_weight=None).
from sklearn.base import clone

modelo_none = make_pipeline(
    clone(preprocesamiento),
    LogisticRegression(class_weight=None, max_iter=1000, random_state=semilla)
)
modelo_none.fit(X_train, y_train)
proba_none = modelo_none.predict_proba(X_valid)[:, 1]
pred_none = (proba_none >= 0.5).astype(int)
print('class_weight=None -> probabilidad máxima:', round(proba_none.max(), 2),
      '| tardías predichas:', int(pred_none.sum()),
      '| Recall:', round(recall_score(y_valid, pred_none, zero_division=0), 3),
      '| ROC-AUC:', round(roc_auc_score(y_valid, proba_none), 3))


class_weight=None -> probabilidad máxima: 0.42 | tardías predichas: 0 | Recall: 0.0 | ROC-AUC: 0.706


### Autoverificación

Ejecute estas comprobaciones después de completar las actividades.

In [15]:
assert len(muestra) == 35000, 'La muestra debe tener 35000 pedidos.'
assert (len(train), len(valid), len(test)) == (24500, 5250, 5250), (
    'Particiones esperadas: 24500 / 5250 / 5250. Revise la celda de muestra.')
assert set(train['order_id']).isdisjoint(valid['order_id']), 'Train y valid comparten pedidos.'
assert set(train['order_id']).isdisjoint(test['order_id']), 'Train y test comparten pedidos.'
assert set(valid['order_id']).isdisjoint(test['order_id']), 'Valid y test comparten pedidos.'

assert modelo is not None and pred is not None, 'Complete la Actividad 1.'
assert len(pred) == len(y_valid), 'pred debe tener 5250 predicciones de validación.'
assert set(np.unique(np.asarray(pred))).issubset({0, 1}), 'pred debe contener solo clases 0 y 1.'
assert np.array_equal(np.asarray(pred), modelo.predict(X_valid)), (
    'pred no coincide con modelo.predict(X_valid). Verifique que utilizó X_valid.')

assert all(isinstance(v, (float, int, np.number)) for v in medidas.values()), (
    'Complete las métricas de la Actividad 2.')
assert proba is not None, 'Complete la Actividad 2: calcule las probabilidades.'
assert np.asarray(proba).ndim == 1 and len(proba) == len(y_valid), (
    'proba debe contener una probabilidad por pedido de validación.')
assert np.allclose(proba, modelo.predict_proba(X_valid)[:, 1]), (
    'proba no coincide con modelo.predict_proba(X_valid)[:, 1].')
assert np.isclose(medidas['f1'], f1_score(y_valid, pred, zero_division=0)), (
    "medidas['f1'] no coincide. Use f1_score(y_valid, pred, zero_division=0).")
assert np.isclose(medidas['ap'], average_precision_score(y_valid, proba)), (
    "medidas['ap'] no coincide. Use average_precision_score(y_valid, proba).")
assert np.isclose(medidas['roc_auc'], roc_auc_score(y_valid, proba)), (
    "medidas['roc_auc'] no coincide. Use roc_auc_score(y_valid, proba).")

assert isinstance(costo_fp_fn, (int, float, np.number)), (
    'Complete la Actividad 4: costo_fp_fn debe ser numérico (8*FP + 35*FN).')
_, fp_ver, fn_ver, _ = confusion_matrix(y_valid, pred).ravel()
assert np.isclose(costo_fp_fn, 8 * fp_ver + 35 * fn_ver), (
    'El costo no coincide con 8*FP + 35*FN. Revise la Actividad 4.')
print('Autoverificación superada.')


Autoverificación superada.


## 5. Ejemplo guiado de regresión

Con los mismos predictores, estime la duración total de la entrega en días usando `Ridge` y evalúe el error absoluto medio (MAE).

In [16]:
from sklearn.base import clone

# Cada modelo recibe una copia independiente del preprocesamiento.
modelo_reg = make_pipeline(clone(preprocesamiento), Ridge(solver='lsqr'))
modelo_reg.fit(X_train, train['dias_entrega'])
pred_dias = modelo_reg.predict(X_valid)
mae = mean_absolute_error(valid['dias_entrega'], pred_dias)
print('MAE regresión:', round(mae, 2), 'días')

MAE regresión: 5.27 días


1. Compare la regresión logística con el modelo de referencia utilizando Accuracy, Recall y F1. Justifique su elección de `class_weight`.

Respuesta: El modelo de referencia siempre dice «a tiempo». Por eso acierta el 93,2 % de las veces (Accuracy 0,932), pero no detecta ningún retraso: su Recall y su F1 son 0. Esto pasa porque en mi muestra solo el 6,8 % de los pedidos llega tarde, así que el Accuracy engaña.

La regresión logística con class_weight='balanced' tiene un Accuracy menor (0,661), pero detecta 229 de los 355 pedidos tardíos (Recall 0,645) y su F1 sube a 0,205. Elegí 'balanced' porque con None el modelo nunca superó el umbral de 0,5 (su probabilidad más alta fue 0,42), así que no marcó ningún pedido como tardío y su Recall y F1 fueron 0, igual que la referencia. Su ROC-AUC (0,706) es casi igual al de 'balanced' (0,705): ordenan igual de bien los pedidos, pero 'balanced' compensa el desbalance y sí genera alertas.

2. Analice qué tipo de pedidos presenta mayor error según los segmentos estudiados. ¿Cómo se relaciona esto con el costo de falsos positivos y falsos negativos?

Respuesta: Los errores se concentran en ciertos estados y meses. Por estado, los peores son Bahía (BA, 71,1 % de error), Río de Janeiro (RJ, 64,3 %), Espírito Santo (ES, 60,5 %) y Santa Catarina (SC, 52,8 %). El mejor es São Paulo (SP, 18,0 %), que además tiene más pedidos (2.141). Por mes, los peores son agosto (47,6 %), marzo (41,3 %) y julio (39,6 %), y el mejor es diciembre (12,2 %). Algunos estados, como Bahía con 201 pedidos, tienen pocos datos, así que su tasa es menos confiable.

Casi todos esos errores son falsos positivos, es decir, alertas que no se cumplen: hay 1.654 falsos positivos frente a 126 falsos negativos. Agosto lo muestra bien: tuvo solo 17 pedidos tardíos reales entre 544, pero 47,6 % de error, así que casi todo fueron alertas de más.

Con los costos del enunciado, el modelo gasta 17.642 en errores, más que no hacer nada (12.425). La razón es que avisa demasiado. Se puede arreglar pidiéndole más seguridad antes de avisar, es decir, subiendo el umbral, que hoy es 0,5. Probé varios umbrales y con uno cercano a 0,76 el costo baja a 12.242. Bajarlo empeora todo: con 0,19 el costo sube a 35.279.

3. Proponga una mejora concreta para la semana 13. Considere la información del producto, del vendedor, del pago o el tratamiento de variables categóricas. Explique cómo verificará la mejora usando la misma partición de validación.

- Subir el umbral. Hoy el modelo avisa cuando está 50 % seguro de que el pedido llegará tarde, y avisa demasiado. Voy a probar varios umbrales en validación y quedarme con el que tenga menor costo. En mi prueba el mejor fue cerca de 0,76 (costo 12.242). Pero eso solo iguala casi al modelo de referencia ( 12.425), así que no basta.
- Agregar una variable de ruta. Combinar el estado del cliente con el del vendedor (por ejemplo, SP→RJ), porque el modelo se equivoca mucho más en unos estados (Bahía, Río) que en otros (São Paulo), y hoy no ve la ruta completa.

Para comprobarlo usaré la misma muestra y la misma validación, sin tocar la prueba. Miraré si el costo baja de 17.642 (lo ideal es que baje de 12.425) y si el AP, que hoy es 0,145, no empeora. Si pasa eso, la mejora funcionó.

## 7. Registro de resultados

Guarde el resultado del laboratorio y la huella de la muestra. Las indicaciones de entrega están en el documento técnico.

In [17]:
import sys
import sklearn

reporte = {
    'laboratorio': 'S12',
    'id_estudiante': identificador,
    'semilla': semilla,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__,
    'n_muestra': len(muestra),
    'n_train': len(train),
    'n_valid': len(valid),
    'n_test': len(test),
    'n_variables': len(columnas),
    'columnas_predictoras': columnas,
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'tasa_tardias_muestra': float(muestra['is_late'].mean()),
    'huella_muestra': huella_muestra,
    'peso_clases': str(peso_clases),
    'metricas': {nombre: float(valor) for nombre, valor in medidas.items()},
    'costo_errores_brl': float(costo_fp_fn),
    'mae_regresion': float(mae),
}

RAIZ = Path.cwd()
while RAIZ != RAIZ.parent and not (RAIZ / '.git').exists():
    RAIZ = RAIZ.parent
if not (RAIZ / '.git').exists():  # sin repositorio (por ejemplo, Colab): carpeta actual
    RAIZ = Path.cwd()
ruta = RAIZ / 'reports' / 'S12' / f'S12_resultados_ID_{identificador}.json'
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reporte:', ruta)

# En Colab, descarga el JSON para conservarlo fuera de la sesión.
try:
    from google.colab import files
    files.download(str(ruta))
except Exception:
    print('Si no se descargó automáticamente, guarde manualmente:', ruta)


Reporte: C:\Users\cifue\ml_challenge_da2026_26\reports\S12\S12_resultados_ID_26.json
Si no se descargó automáticamente, guarde manualmente: C:\Users\cifue\ml_challenge_da2026_26\reports\S12\S12_resultados_ID_26.json


C:\Users\cifue\AppData\Local\Temp\ipykernel_12596\268688921.py:8: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
